# 🏛️ Palghar AI — RAG + Vector Database (Marathi PDFs) with a Streamlit web app

**What you get at the end:** a public web page where officials can **upload or choose PDFs (Marathi / Hindi / English, scanned or digital)**, then **ask questions in Marathi, Hindi or English**. Every answer is written **only from the documents** and shows its **sources** (file name, page number, match %, and the exact passage). A separate page shows the **specification of the running models and the vector database**.

### How this follows the faculty's RAG course (Day 1 – Day 4)
| Faculty topic | Used here |
|---|---|
| Text extraction from PDF (Day 2) | PyMuPDF, plus **OCR (Tesseract, Marathi)** for scanned pages |
| Chunking (Day 2) | Sentence-aware chunks that understand the Marathi full stop **।** |
| Embeddings with Sentence-Transformers (Day 1–2) | **`intfloat/multilingual-e5-base`** (the course used English-only MiniLM; that cannot search Marathi well) |
| ChromaDB persistent vector database (Day 3) | ChromaDB `PersistentClient`, cosine distance, one collection per embedding model |
| Similarity threshold, grounded prompt, Gemini (Day 3) | Minimum-match slider + "answer only from context" system prompt + Gemini |
| Source + page metadata, chat history, question rewriting (Day 4) | Source cards on every answer, conversation memory, follow-up question rewriting |

### Do I have to train again after Colab reconnects?
**No training exists in RAG.** Your PDFs are indexed **once** (reading + embeddings + vector DB). The index is saved in one storage folder. Colab wipes its disk when it disconnects, so keep that folder safe with **either**:
* **Option A (no Google Drive):** in the web page, *Knowledge Base → Backup and restore → Download backup*. After a reconnect set `RESTORE_BACKUP = True` in Step 2 (or use *Restore* in the web page) and upload that ZIP.
* **Option B (Google Drive):** set `USE_GOOGLE_DRIVE = True` in Step 2. Everything (PDFs, index, even the embedding model) is stored in `MyDrive/PalgharAI` and is picked up automatically next time.

### Run
1. **Runtime → Change runtime type → T4 GPU** (faster embeddings; CPU also works).
2. *(Recommended)* Add your Gemini key in Colab **Secrets** (🔑 icon on the left) with the name `GEMINI_API_KEY` and enable notebook access. Without a key the app still works in *retrieval-only* mode (shows the matching passages with sources, no written answer).
3. **Runtime → Run all.** Open the link printed by the last cell.
4. In the web page: **Knowledge Base → upload PDFs → Index**. Then go to **Ask Palghar AI**.

## Step 1 — Install packages
Installs Tesseract OCR with **Marathi** language data, and the Python libraries. PyTorch is already in Colab and is not touched.

In [1]:
import subprocess, sys

# Tesseract OCR + Marathi, Hindi and English language data (for scanned PDFs)
subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-mar tesseract-ocr-hin tesseract-ocr-eng",
               shell=True, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "chromadb", "sentence-transformers", "pymupdf", "pytesseract", "pillow",
                       "google-genai", "streamlit", "pandas"])
print("Installed.")
print(subprocess.run("tesseract --list-langs", shell=True, capture_output=True, text=True).stdout.strip().replace("\n", ", "))

Installed.
List of available languages in "/usr/share/tesseract-ocr/5/tessdata/" (4):, eng, hin, mar, osd


## Step 2 — Settings

In [2]:
# ---------------- where the saved index lives ----------------
USE_GOOGLE_DRIVE = True      # True  -> keep everything in MyDrive/PalgharAI (survives reconnects automatically)
RESTORE_BACKUP   = False      # True  -> upload a backup ZIP now (use after a reconnect when USE_GOOGLE_DRIVE = False)

# ---------------- models ----------------
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"   # multilingual (Marathi, Hindi, English). Lighter: "intfloat/multilingual-e5-small"
GEMINI_MODEL    = "gemini-3.5-flash-lite"           # model used in the faculty's Day 4 notebook. Change if your key lists another one.
ASK_FOR_KEY     = True        # if no Colab Secret is found, ask for the Gemini key (press Enter to skip -> retrieval-only mode)

PORT = 8504

## Step 3 — RAG engine (`palghar_rag.py`)
PDF reading + OCR, chunking, embeddings, ChromaDB, retrieval, grounded prompt, Gemini. Both this notebook and the web app use this one file.

In [3]:
%%writefile /content/palghar_rag.py
"""
palghar_rag.py - core of Palghar AI (RAG + vector database).

Pipeline (same building blocks the faculty taught, Day 1-4):
  PDF -> text (PyMuPDF, OCR for scanned Marathi pages) -> chunks -> multilingual embeddings
  -> ChromaDB (persistent) -> semantic search -> grounded prompt -> Gemini -> answer + sources
"""
import hashlib
import io
import json
import re
import shutil
import time
import unicodedata
import zipfile
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_EMBED_MODEL = "intfloat/multilingual-e5-base"
DEFAULT_GEMINI_MODEL = "gemini-3.5-flash-lite"


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


# ------------------------------------------------------------------ storage (everything that must survive a reconnect)
class Storage:
    def __init__(self, root):
        self.root = Path(root)
        self.pdfs = self.root / "pdfs"
        self.db = self.root / "chroma_db"
        self.registry_path = self.root / "registry.json"
        for p in (self.root, self.pdfs, self.db):
            p.mkdir(parents=True, exist_ok=True)

    def load_registry(self):
        if self.registry_path.exists():
            try:
                return json.loads(self.registry_path.read_text(encoding="utf-8"))
            except Exception:
                return {}
        return {}

    def save_registry(self, reg):
        self.registry_path.write_text(json.dumps(reg, ensure_ascii=False, indent=2), encoding="utf-8")

    def save_pdf(self, file_hash, filename, data):
        safe = re.sub(r"[^\w.\- ()]+", "_", Path(filename).name, flags=re.UNICODE)
        path = self.pdfs / f"{file_hash[:8]}__{safe}"
        path.write_bytes(data)
        return path

    def pdf_path(self, file_hash):
        hits = list(self.pdfs.glob(f"{file_hash[:8]}__*"))
        return hits[0] if hits else None

    def make_backup(self, zip_path):
        zip_path = Path(zip_path)
        if zip_path.exists():
            zip_path.unlink()
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for f in self.root.rglob("*"):
                if f.is_file() and f.resolve() != zip_path.resolve():
                    z.write(f, arcname=str(f.relative_to(self.root)))
        return zip_path

    def restore_backup(self, zip_bytes_or_path):
        src = io.BytesIO(zip_bytes_or_path) if isinstance(zip_bytes_or_path, (bytes, bytearray)) else zip_bytes_or_path
        with zipfile.ZipFile(src) as z:
            names = z.namelist()
            if "registry.json" not in names and not any(n.startswith("chroma_db/") for n in names):
                raise ValueError("This ZIP does not look like a Palghar AI backup.")
            z.extractall(self.root)
        return len(names)


# ------------------------------------------------------------------ PDF text (with OCR fallback for scanned Marathi pages)
def normalize_text(text):
    text = (text or "").replace("\x00", "").replace("\r\n", "\n").replace("\r", "\n")
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def ocr_available():
    try:
        import pytesseract
        langs = pytesseract.get_languages(config="")
        return True, langs
    except Exception:
        return False, []


def ocr_page(page, lang="mar+eng", dpi=200):
    import pytesseract
    from PIL import Image
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(img, lang=lang)


def extract_pages(pdf_path, force_ocr=False, ocr_lang="mar+eng", min_chars=40):
    """Returns (pages, warnings). Each page: {page, text, ocr}."""
    import fitz
    doc = fitz.open(str(pdf_path))
    pages, warnings, ocr_ok = [], [], None
    for i, page in enumerate(doc):
        text = "" if force_ocr else normalize_text(page.get_text("text", sort=True))
        used_ocr = False
        if force_ocr or len(text) < min_chars:
            if ocr_ok is None:
                ocr_ok, langs = ocr_available()
                if not ocr_ok:
                    warnings.append("OCR is not available (tesseract missing); scanned pages were skipped.")
                elif "mar" not in langs:
                    warnings.append("Marathi OCR data (tesseract-ocr-mar) is not installed; OCR will use English only.")
                    ocr_lang = "eng"
            if ocr_ok:
                try:
                    ocr_text = normalize_text(ocr_page(page, ocr_lang))
                    if len(ocr_text) > len(text):
                        text, used_ocr = ocr_text, True
                except Exception as exc:
                    warnings.append(f"OCR failed on page {i + 1}: {exc}")
        pages.append({"page": i + 1, "text": text, "ocr": used_ocr})
    doc.close()
    return pages, warnings


# ------------------------------------------------------------------ chunking (sentence-aware; Marathi danda aware)
def chunk_text(text, size=700, overlap=120):
    text = re.sub(r"[ \t]+", " ", text or "").strip()
    if not text:
        return []
    parts = [p.strip() for p in re.split(r"(?<=[।॥.?!])\s+|\n+", text) if p.strip()]
    def word_tail(s):                                      # last ~overlap chars, starting at a word boundary
        if not s or not overlap or len(s) <= overlap:
            return ""
        t = s[-overlap:]
        i = t.find(" ")
        return t[i + 1:] if i != -1 else ""

    def hard_cut(s):                                       # cut a very long unbroken text at a space
        cut = s.rfind(" ", 0, size)
        cut = cut if cut > size * 0.5 else size
        return s[:cut].strip(), s[max(1, cut - overlap):].lstrip() if overlap and cut > overlap else s[cut:].lstrip()

    chunks, cur = [], ""
    for sent in parts:
        if len(cur) + len(sent) + 1 <= size:
            cur = f"{cur} {sent}".strip()
            continue
        if cur:
            chunks.append(cur)
        tail = word_tail(cur)
        cur = f"{tail} {sent}".strip() if len(sent) < size else sent
        while len(cur) > int(size * 1.5):
            head, cur = hard_cut(cur)
            chunks.append(head)
    if cur:
        chunks.append(cur)
    return chunks


# ------------------------------------------------------------------ embeddings (multilingual: Marathi / Hindi / English)
class Embedder:

# ------------------------------------------------------------------
# Embeddings (direct Transformers loading - no SentenceTransformer
# AutoProcessor initialization)
    def __init__(self, name=DEFAULT_EMBED_MODEL, device=None, encode_fn=None):
        import os
        import torch
        from transformers import AutoTokenizer, AutoModel

        self.name = name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._fn = encode_fn
        self.model = None
        self.tokenizer = None
        self.dim = None
        self.max_seq_length = 512
        self.e5 = "e5" in name.lower()

        if encode_fn is None:
            cache_dir = os.environ.get("HF_HOME")

            print(f"Loading tokenizer: {name}")

            self.tokenizer = AutoTokenizer.from_pretrained(
                name,
                cache_dir=cache_dir,
                use_fast=True
            )

            print(f"Loading transformer model: {name}")

            self.model = AutoModel.from_pretrained(
                name,
                cache_dir=cache_dir,
                use_safetensors=True
            )

            self.model.to(self.device)
            self.model.eval()

            self.dim = self.model.config.hidden_size

            print("Embedding model loaded successfully!")
            print("Device:", self.device)
            print("Vector dimension:", self.dim)

    def _encode(self, texts, batch_size=16):
        if self._fn is not None:
            return self._fn(texts)

        import torch
        import torch.nn.functional as F

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]

            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_seq_length,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(self.device)
                for key, value in inputs.items()
            }

            with torch.inference_mode():
                outputs = self.model(**inputs)

                token_embeddings = outputs.last_hidden_state

                attention_mask = inputs["attention_mask"]
                mask = attention_mask.unsqueeze(-1).expand(
                    token_embeddings.size()
                ).to(token_embeddings.dtype)

                # Mean pooling
                summed = torch.sum(token_embeddings * mask, dim=1)
                counts = torch.sum(mask, dim=1).clamp(min=1e-9)

                embeddings = summed / counts

                # Normalize vectors
                embeddings = F.normalize(
                    embeddings,
                    p=2,
                    dim=1
                )

                all_embeddings.extend(
                    embeddings.cpu().tolist()
                )

        return all_embeddings

    def embed_passages(self, texts, batch_size=16):
        texts = [
            ("passage: " + t) if self.e5 else t
            for t in texts
        ]
        return self._encode(texts, batch_size)

    def embed_query(self, text):
        query = ("query: " + text) if self.e5 else text
        return self._encode([query])[0]

    def spec(self):
        return {
            "name": self.name,
            "dimension": self.dim,
            "max_tokens": self.max_seq_length,
            "device": self.device,
            "query_prefix": "query: / passage:" if self.e5 else "none"
        }



# ------------------------------------------------------------------ vector database (ChromaDB, persistent)
class VectorStore:
    def __init__(self, storage, embed_name):
        import chromadb
        self.chromadb_version = chromadb.__version__
        self.path = str(storage.db)
        self.client = chromadb.PersistentClient(path=self.path)
        slug = re.sub(r"[^a-zA-Z0-9]+", "_", embed_name).strip("_")[-40:] or "model"
        self.name = f"palghar_{slug}"          # one collection per embedding model (vectors are not interchangeable)
        try:
            self.col = self.client.get_or_create_collection(name=self.name, metadata={"hnsw:space": "cosine"})
        except Exception:
            self.col = self.client.get_or_create_collection(name=self.name, configuration={"hnsw": {"space": "cosine"}})

    def count(self):
        return self.col.count()

    def add_chunks(self, file_hash, source, items, embeddings):
        ids = [f"{file_hash[:12]}_{it['page']}_{it['chunk']}" for it in items]
        self.col.upsert(
            ids=ids,
            documents=[it["text"] for it in items],
            embeddings=embeddings,
            metadatas=[{"file_hash": file_hash, "source": source, "page": it["page"],
                        "chunk": it["chunk"], "ocr": bool(it["ocr"])} for it in items],
        )

    def delete_document(self, file_hash):
        try:
            self.col.delete(where={"file_hash": file_hash})
        except Exception:
            pass

    def query(self, query_embedding, k=5, file_hashes=None):
        if self.count() == 0:
            return []
        kwargs = {"query_embeddings": [query_embedding], "n_results": min(k, self.count()),
                  "include": ["documents", "metadatas", "distances"]}
        if file_hashes:
            kwargs["where"] = {"file_hash": {"$in": list(file_hashes)}}
        res = self.col.query(**kwargs)
        out = []
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
            out.append({"text": doc, "source": meta.get("source", "?"), "page": meta.get("page", "?"),
                        "ocr": bool(meta.get("ocr")), "file_hash": meta.get("file_hash"),
                        "similarity": round(1.0 - float(dist), 4)})
        return out


# ------------------------------------------------------------------ ingestion (index once, reuse after every reconnect)
def ingest_pdf(storage, store, embedder, data, filename, force_ocr=False, reindex=False,
               chunk_size=700, overlap=120, progress=None):
    """Index one PDF. Skips work if this exact file is already indexed with this embedding model."""
    say = progress or (lambda msg, frac=None: None)
    file_hash = hashlib.sha256(data).hexdigest()
    reg = storage.load_registry()
    entry = reg.get(file_hash, {})
    if entry and embedder.name in entry.get("indexed", {}) and not reindex:
        return {"status": "already_indexed", "name": entry["name"], "hash": file_hash,
                "chunks": entry["indexed"][embedder.name]["chunks"], "warnings": []}

    path = storage.pdf_path(file_hash) or storage.save_pdf(file_hash, filename, data)
    say(f"Reading {filename} ...", 0.05)
    pages, warnings = extract_pages(path, force_ocr=force_ocr)
    items = []
    for p in pages:
        for j, ch in enumerate(chunk_text(p["text"], chunk_size, overlap)):
            items.append({"page": p["page"], "chunk": j, "text": ch, "ocr": p["ocr"]})
    chars = sum(len(p["text"]) for p in pages)
    if not items:
        return {"status": "no_text", "name": filename, "hash": file_hash, "chunks": 0,
                "warnings": warnings + ["No readable text found. If this is a scanned PDF, enable OCR."]}

    say(f"{len(pages)} pages, {len(items)} passages. Creating embeddings ...", 0.25)
    store.delete_document(file_hash)
    embeddings = []
    step = 32
    for i in range(0, len(items), step):
        embeddings += embedder.embed_passages([it["text"] for it in items[i:i + step]], step)
        say(f"Embedding passages {min(i + step, len(items))}/{len(items)}", 0.25 + 0.65 * min(1, (i + step) / len(items)))
    store.add_chunks(file_hash, Path(filename).name, items, embeddings)

    entry = reg.get(file_hash, {})
    entry.update({"name": Path(filename).name, "size": len(data), "pages": len(pages), "chars": chars,
                  "ocr_pages": sum(1 for p in pages if p["ocr"]), "uploaded_at": entry.get("uploaded_at", now_iso())})
    entry.setdefault("indexed", {})[embedder.name] = {"chunks": len(items), "at": now_iso()}
    reg[file_hash] = entry
    storage.save_registry(reg)
    say("Done", 1.0)
    return {"status": "indexed", "name": filename, "hash": file_hash, "chunks": len(items),
            "pages": len(pages), "ocr_pages": entry["ocr_pages"], "warnings": warnings}


def remove_document(storage, store, file_hash, delete_file=True):
    store.delete_document(file_hash)
    reg = storage.load_registry()
    reg.pop(file_hash, None)
    storage.save_registry(reg)
    if delete_file:
        p = storage.pdf_path(file_hash)
        if p:
            p.unlink(missing_ok=True)


# ------------------------------------------------------------------ retrieval + prompt (grounded, with numbered sources)
SYSTEM_PROMPT = (
    "You are Palghar AI, a district information assistant for government officials of Palghar, Maharashtra.\n"
    "Rules:\n"
    "1. Answer ONLY from the numbered context passages. Do not use outside knowledge and never guess.\n"
    "2. Cite the passage number after each fact, like [1] or [2][3].\n"
    "3. If the context does not contain the answer, say clearly that you could not find it in the uploaded "
    "documents. Do not invent figures.\n"
    "4. Keep numbers, units, years and names exactly as written in the passages. If passages disagree, say so.\n"
    "5. Reply in the same language as the question (Marathi, Hindi or English). Be concise and clear."
)


def format_context(passages):
    return "\n\n".join(f"[{i}] (Source: {p['source']}, page {p['page']})\n{p['text']}"
                       for i, p in enumerate(passages, 1))


def build_prompt(question, passages, history=None):
    hist = ""
    for m in (history or [])[-6:]:
        hist += f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content'][:500]}\n"
    ctx = format_context(passages) if passages else "(no relevant passages found)"
    return (f"Previous conversation:\n{hist or '(none)'}\n\nContext passages:\n{ctx}\n\n"
            f"Question: {question}\n\nAnswer (with [n] citations):")


def build_rewrite_prompt(question, history):
    hist = "".join(f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content'][:400]}\n" for m in history[-6:])
    return ("Rewrite the latest question as a complete standalone question, using the conversation for context. "
            "Keep the same language. Do not answer it. Return only the rewritten question.\n\n"
            f"Conversation:\n{hist}\nLatest question: {question}\n\nStandalone question:")


def retrieve(store, embedder, query, k=5, min_similarity=0.0, file_hashes=None):
    qv = embedder.embed_query(query)
    hits = store.query(qv, k=k, file_hashes=file_hashes)
    return [h for h in hits if h["similarity"] >= min_similarity], hits


# ------------------------------------------------------------------ Gemini (as taught in Day 3 / Day 4)
class GeminiLLM:
    def __init__(self, api_key, model=DEFAULT_GEMINI_MODEL, temperature=0.2, max_output_tokens=1500):
        from google import genai
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def _config(self, system):
        cfg = {"temperature": self.temperature, "max_output_tokens": self.max_output_tokens}
        if system:
            cfg["system_instruction"] = system
        return cfg

    def generate(self, prompt, system=None):
        r = self.client.models.generate_content(model=self.model, contents=prompt, config=self._config(system))
        return (r.text or "").strip()

    def stream(self, prompt, system=None):
        for chunk in self.client.models.generate_content_stream(model=self.model, contents=prompt,
                                                                config=self._config(system)):
            if getattr(chunk, "text", None):
                yield chunk.text

    def spec(self):
        return {"model": self.model, "temperature": self.temperature, "max_output_tokens": self.max_output_tokens}

    def list_models(self):
        names = []
        for m in self.client.models.list():
            actions = getattr(m, "supported_actions", None) or []
            if "generateContent" in actions or not actions:
                names.append(m.name.replace("models/", ""))
        return sorted(n for n in names if n.startswith("gemini"))


def timed(fn, *a, **kw):
    t = time.perf_counter()
    out = fn(*a, **kw)
    return out, (time.perf_counter() - t) * 1000.0

Writing /content/palghar_rag.py


## Step 4 — Storage folder (keeps your index across reconnects)

In [4]:
import os, sys
from pathlib import Path
sys.path.insert(0, "/content")
import importlib, palghar_rag
importlib.reload(palghar_rag)
from palghar_rag import Storage

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_DIR = "/content/drive/MyDrive/Palghar_AI"
else:
    STORAGE_DIR = "/content/PalgharAI_storage"

storage = Storage(STORAGE_DIR)

# Keep the downloaded embedding model next to the index when using Drive (no re-download after reconnect)
HF_CACHE = str(Path(STORAGE_DIR) / "hf_cache") if USE_GOOGLE_DRIVE else str(Path.home() / ".cache" / "huggingface")
os.environ["HF_HOME"] = HF_CACHE

if RESTORE_BACKUP:
    from google.colab import files
    print("Select your PalgharAI_backup.zip")
    up = files.upload()
    for name, data in up.items():
        print(f"Restored {storage.restore_backup(data)} files from {name}")

reg = storage.load_registry()
print("Storage folder :", STORAGE_DIR)
if reg:
    print(f"Existing index found: {len(reg)} document(s) -> no re-indexing needed")
    for e in reg.values():
        print("  -", e["name"], f"({e['pages']} pages)")
else:
    print("No documents indexed yet. Use the web page (Knowledge Base) after launching.")

Mounted at /content/drive
Storage folder : /content/drive/MyDrive/Palghar_AI
Existing index found: 9 document(s) -> no re-indexing needed
  - Text_to_PDF_Onlinenotpad (1).pdf (29 pages)
  - ruralwater.pdf (5 pages)
  - MH34-Palghar-07.05.2016.pdf (49 pages)
  - health.pdf (6 pages)
  - district rural.pdf (4 pages)
  - budget.pdf (42 pages)
  - 17619016804475.pdf (91 pages)
  - 2025041138.pdf (141 pages)
  - 2019073010.pdf (60 pages)


## Step 5 — The web app (`app.py`) and its dark theme
Pages: **Ask Palghar AI** (chat with sources) · **Knowledge Base** (upload / choose / index / delete / backup) · **Model & System** (specification).

In [5]:
%%writefile /content/app.py
import html
import os
import platform
import sys
import time
from pathlib import Path

import pandas as pd
import streamlit as st

sys.path.insert(0, str(Path(__file__).resolve().parent))
import palghar_rag as rag  # noqa: E402

STORAGE_DIR = os.environ.get("PALGHAR_STORAGE", "/content/PalgharAI_storage")
EMBED_NAME = os.environ.get("PALGHAR_EMBED", rag.DEFAULT_EMBED_MODEL)
DEFAULT_MODEL = os.environ.get("GEMINI_MODEL", rag.DEFAULT_GEMINI_MODEL)

st.set_page_config(page_title="Palghar AI", page_icon="🏛️", layout="wide", initial_sidebar_state="expanded")

# ------------------------------------------------------------------ design system
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Noto+Sans+Devanagari:wght@400;600;700&display=swap');
html, body, [class*="css"], .stApp { font-family: 'Inter', 'Noto Sans Devanagari', sans-serif; }
.stApp { background: radial-gradient(1200px 500px at 85% -10%, rgba(23,195,178,.12), transparent), #0B1220; }
[data-testid="stSidebar"] { background: #0E1A2E; border-right: 1px solid #1E3150; }
.block-container { padding-top: 1.6rem; max-width: 1180px; }
.hero { background: linear-gradient(120deg, #0F3A4A 0%, #14284A 60%, #1B2A55 100%); border: 1px solid #25507A;
        border-radius: 20px; padding: 28px 32px; margin-bottom: 18px; position: relative; overflow: hidden; }
.hero:after { content:""; position:absolute; right:-60px; top:-60px; width:260px; height:260px; border-radius:50%;
              background: radial-gradient(circle, rgba(242,163,58,.35), transparent 70%); }
.hero h1 { margin:0; font-size: 38px; font-weight: 700; color:#F6F4EE; letter-spacing:-.5px; }
.hero p { margin:6px 0 14px; color:#B9C9DC; font-size:16px; }
.chip { display:inline-block; padding:5px 12px; margin:0 8px 6px 0; border-radius:999px; font-size:12.5px; font-weight:600;
        background: rgba(23,195,178,.14); border:1px solid rgba(23,195,178,.45); color:#5EEAD4; }
.chip.warn { background: rgba(242,163,58,.14); border-color: rgba(242,163,58,.5); color:#F2C36B; }
.chip.off { background: rgba(148,163,184,.12); border-color: rgba(148,163,184,.35); color:#94A3B8; }
.card { background:#111C2E; border:1px solid #223656; border-radius:16px; padding:18px 20px; height:100%; }
.card h4 { margin:0 0 4px; color:#F6F4EE; font-size:16px; }
.card .sub { color:#8FA3BC; font-size:12.5px; margin-bottom:10px; }
.kv { display:flex; justify-content:space-between; gap:12px; padding:6px 0; border-bottom:1px dashed #223656; font-size:13.5px; }
.kv span:first-child { color:#8FA3BC; } .kv span:last-child { color:#E5E7EB; font-weight:600; text-align:right; word-break:break-all; }
.stat { background:#111C2E; border:1px solid #223656; border-radius:14px; padding:14px 16px; }
.stat b { display:block; font-size:26px; color:#5EEAD4; line-height:1.1; } .stat span { color:#8FA3BC; font-size:12.5px; }
.src { background:#0F1A2C; border:1px solid #223656; border-left:4px solid #F2A33A; border-radius:12px; padding:12px 14px; margin:8px 0; }
.src.low { border-left-color:#64748B; }
.src-top { display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
.src-n { background:#F2A33A; color:#0B1F3A; font-weight:700; border-radius:50%; width:22px; height:22px; display:inline-flex;
         align-items:center; justify-content:center; font-size:12px; }
.src-name { color:#F6F4EE; font-weight:600; font-size:13.5px; }
.tag { background:#1B2D4A; color:#9DB4D2; border-radius:6px; padding:2px 8px; font-size:11.5px; font-weight:600; }
.src-sim { margin-left:auto; color:#5EEAD4; font-weight:700; font-size:13px; }
.bar { height:5px; background:#1B2D4A; border-radius:4px; margin:8px 0; } .bar div { height:5px; border-radius:4px; background:linear-gradient(90deg,#17C3B2,#F2A33A); }
.src details summary { cursor:pointer; color:#B9C9DC; font-size:13.5px; line-height:1.5; }
.src details div { color:#D5DEE8; font-size:13.5px; line-height:1.6; margin-top:8px; padding-top:8px; border-top:1px solid #223656; }
.meta { color:#8FA3BC; font-size:12px; margin-top:6px; }
.step { background:#111C2E; border:1px solid #223656; border-radius:12px; padding:12px 8px; text-align:center; font-size:13px; color:#E5E7EB; }
.step b { display:block; color:#F2A33A; font-size:12px; margin-bottom:2px; }
.notice { background:rgba(242,163,58,.08); border:1px solid rgba(242,163,58,.4); border-radius:12px; padding:14px 16px; color:#F2D9A8; font-size:14px; }
div[data-testid="stChatMessage"] { background:#0F1A2C; border:1px solid #1E3150; border-radius:14px; padding:12px 14px; }
.stButton button, .stDownloadButton button { border-radius:10px; border:1px solid #2A4A73; font-weight:600; }
.stButton button[kind="primary"] { background:#0D9488; border-color:#0D9488; }
</style>
""", unsafe_allow_html=True)


# ------------------------------------------------------------------ cached resources
@st.cache_resource(show_spinner=False)
def get_storage(path):
    return rag.Storage(path)


@st.cache_resource(show_spinner=False)
def get_embedder(name):
    return rag.Embedder(name)


@st.cache_resource(show_spinner=False)
def get_store(path, embed_name):
    return rag.VectorStore(get_storage(path), embed_name)


@st.cache_resource(show_spinner=False)
def get_llm(api_key, model, temperature):
    return rag.GeminiLLM(api_key, model=model, temperature=temperature)


storage = get_storage(STORAGE_DIR)
store = get_store(STORAGE_DIR, EMBED_NAME)


def embedder_ready():
    try:
        return get_embedder(EMBED_NAME)
    except Exception as exc:
        st.error(f"Embedding model could not be loaded: {type(exc).__name__}: {exc}")
        st.stop()


# ------------------------------------------------------------------ sidebar
st.session_state.setdefault("page", "💬 Ask Palghar AI")
st.session_state.setdefault("messages", [])
st.session_state.setdefault("uploader_key", 0)

with st.sidebar:
    st.markdown("## 🏛️ Palghar AI")
    st.caption("District Development Intelligence Platform")
    st.radio("Navigate", ["💬 Ask Palghar AI", "📚 Knowledge Base", "⚙️ Model & System"], key="page",
             label_visibility="collapsed")
    st.divider()

    st.markdown("**Gemini (answer generator)**")
    default_key = os.environ.get("GEMINI_API_KEY", "")
    typed_key = st.text_input("API key", type="password", value="", placeholder="set in notebook" if default_key else "paste key",
                              help="Optional here if the notebook already loaded GEMINI_API_KEY from Colab Secrets.")
    api_key = typed_key.strip() or default_key
    model_name = st.text_input("Model", value=DEFAULT_MODEL)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.2, 0.1)
    llm = None
    if api_key:
        try:
            llm = get_llm(api_key, model_name, temperature)
        except Exception as exc:
            st.error(f"Gemini setup failed: {exc}")
    st.markdown(f'<span class="chip {"" if llm else "warn"}">{"Gemini connected" if llm else "Retrieval-only mode"}</span>',
                unsafe_allow_html=True)
    reg0 = storage.load_registry()
    st.markdown(f'<span class="chip {"" if store.count() else "off"}">{len(reg0)} docs · {store.count()} passages</span>',
                unsafe_allow_html=True)
    st.caption("Index is saved on disk. After a Colab reconnect nothing needs to be re-trained or re-indexed.")


def hero(title_sub):
    llm_chip = '<span class="chip">Gemini answers</span>' if llm else '<span class="chip warn">Retrieval-only (no API key)</span>'
    st.markdown(f"""
    <div class="hero"><h1>Palghar AI</h1><p>{html.escape(title_sub)}</p>
    <span class="chip">RAG</span><span class="chip">ChromaDB vector database</span>
    <span class="chip">मराठी · हिंदी · English</span><span class="chip">Sources on every answer</span>{llm_chip}</div>
    """, unsafe_allow_html=True)


def source_cards(passages, low=False):
    out = []
    for i, p in enumerate(passages, 1):
        sim = max(0.0, min(1.0, p["similarity"]))
        text = html.escape(p["text"]).replace("\n", "<br>")
        summary = html.escape(p["text"][:170].replace("\n", " ")) + ("…" if len(p["text"]) > 170 else "")
        ocr = '<span class="tag">OCR</span>' if p.get("ocr") else ""
        out.append(
            f'<div class="src {"low" if low else ""}"><div class="src-top"><span class="src-n">{i}</span>'
            f'<span class="src-name">{html.escape(str(p["source"]))}</span><span class="tag">Page {p["page"]}</span>{ocr}'
            f'<span class="src-sim">{sim:.0%} match</span></div><div class="bar"><div style="width:{sim * 100:.0f}%"></div></div>'
            f'<details><summary>{summary}</summary><div>{text}</div></details></div>')
    return "".join(out)


def render_sources(msg):
    srcs = msg.get("sources") or []
    if not srcs:
        st.markdown('<div class="meta">Sources: none retrieved (the knowledge base is empty).</div>', unsafe_allow_html=True)
        return
    label = "Closest passages (below the match threshold)" if msg.get("low") else f"Sources ({len(srcs)})"
    with st.expander(f"📎 {label}", expanded=not msg.get("low")):
        st.markdown(source_cards(srcs, low=msg.get("low", False)), unsafe_allow_html=True)
    m = msg.get("meta", {})
    if m:
        st.markdown(f'<div class="meta">Search {m.get("retrieval_ms", 0):.0f} ms · Answer {m.get("generation_ms", 0):.0f} ms · '
                    f'{len(srcs)} passages · mode: {m.get("mode", "")}</div>', unsafe_allow_html=True)


# ================================================================== PAGE 1: ASK
def page_ask():
    hero("Ask questions about Palghar district in Marathi, Hindi or English. Every answer shows where it came from.")
    reg = storage.load_registry()
    names = {h: e["name"] for h, e in reg.items() if EMBED_NAME in e.get("indexed", {})}

    if store.count() == 0:
        st.markdown('<div class="notice"><b>No documents indexed yet.</b> Open <b>Knowledge Base</b>, upload your PDFs '
                    '(Marathi or English) and press <b>Index</b>. This takes a few minutes once; the index is then saved.</div>',
                    unsafe_allow_html=True)
        st.button("Go to Knowledge Base →", on_click=lambda: st.session_state.update(page="📚 Knowledge Base"))
        return

    with st.expander("⚙️ Search settings"):
        c1, c2, c3 = st.columns(3)
        k = c1.slider("Passages to retrieve", 1, 10, 5)
        min_sim = c2.slider("Minimum match", 0.0, 1.0, 0.0, 0.05,
                            help="Passages below this similarity are not given to the model. e5 scores are compressed, typically 0.7–0.9.")
        rewrite = c3.checkbox("Understand follow-up questions", value=True,
                              help="Rewrites a follow-up into a standalone question first (Day 4 technique). Needs Gemini.")
        scope = st.multiselect("Search only in these documents (empty = all)", options=list(names.values()))
    scope_hashes = [h for h, n in names.items() if n in scope] or None

    question = None
    if not st.session_state.messages:
        st.markdown("##### Try a question")
        sugg = ["पालघर जिल्ह्यातील आरोग्य सुविधांबद्दल सांगा.", "शेती आणि सिंचनाबद्दल माहिती द्या.",
                "What are the major development challenges in Palghar?", "Summarise education in Palghar district."]
        cols = st.columns(2)
        for i, s in enumerate(sugg):
            if cols[i % 2].button(s, key=f"sg{i}"):
                question = s

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
            if msg["role"] == "assistant":
                render_sources(msg)

    typed = st.chat_input("प्रश्न विचारा / Ask a question about Palghar...")
    question = typed or question
    if not question:
        if st.session_state.messages and st.button("🗑️ Clear conversation"):
            st.session_state.messages = []
            st.rerun()
        return

    history = [m for m in st.session_state.messages if m["role"] in ("user", "assistant")]
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        emb = embedder_ready()
        search_q = question
        if rewrite and history and llm:
            try:
                rq, _ = rag.timed(llm.generate, rag.build_rewrite_prompt(question, history))
                search_q = rq.strip().splitlines()[0] if rq.strip() else question
                st.caption(f"Searching for: {search_q}")
            except Exception:
                search_q = question
        (passages, closest), t_ret = rag.timed(rag.retrieve, store, emb, search_q, k, min_sim, scope_hashes)

        reply, t_gen, mode, low = "", 0.0, "retrieval-only", False
        if not passages:
            reply = ("I could not find this in the uploaded documents. Try rephrasing, lowering the minimum match, "
                     "or adding the relevant PDF in the Knowledge Base.")
            low, mode = True, "no match"
        elif llm is None:
            reply = ("**Retrieval-only mode** (no Gemini key). These are the most relevant passages from the documents. "
                     "Add a key in the sidebar to get a written answer.")
        else:
            prompt = rag.build_prompt(search_q, passages, history)
            t0 = time.perf_counter()
            try:
                reply = st.write_stream(llm.stream(prompt, rag.SYSTEM_PROMPT))
                mode = f"RAG · {llm.model}"
            except Exception as exc:
                reply = (f"Gemini could not answer ({type(exc).__name__}: {str(exc)[:200]}). "
                         "Check the API key and model name in the sidebar. Showing the retrieved passages instead.")
                st.markdown(reply)
                mode = "retrieval-only (LLM error)"
            t_gen = (time.perf_counter() - t0) * 1000
        if mode.startswith("retrieval") or mode == "no match":
            st.markdown(reply)
        msg = {"role": "assistant", "content": reply, "sources": passages if passages else closest, "low": low,
               "meta": {"retrieval_ms": t_ret, "generation_ms": t_gen, "mode": mode}}
        render_sources(msg)
        st.session_state.messages.append(msg)


# ================================================================== PAGE 2: KNOWLEDGE BASE
def page_kb():
    hero("Upload or choose the PDFs Palghar AI should learn from. Marathi and scanned documents are supported.")
    reg = storage.load_registry()
    docs = [e for e in reg.values() if embedder_name_ok(e)]
    c = st.columns(4)
    for col, (val, lab) in zip(c, [(len(docs), "Documents"), (store.count(), "Searchable passages"),
                                   (sum(e["pages"] for e in docs), "Pages read"), (sum(e.get("ocr_pages", 0) for e in docs), "Pages via OCR")]):
        col.markdown(f'<div class="stat"><b>{val}</b><span>{lab}</span></div>', unsafe_allow_html=True)
    st.write("")

    left, right = st.columns([1.1, 1])
    with left:
        st.markdown("#### ⬆️ Upload PDFs")
        files = st.file_uploader("Drop PDF files here (Marathi / English)", type=["pdf"], accept_multiple_files=True,
                                 key=f"up{st.session_state.uploader_key}")
        force_ocr = st.checkbox("Force OCR (use when Marathi text looks broken or the PDF is scanned)")
        with st.expander("Advanced"):
            csize = st.slider("Passage size (characters)", 300, 1500, 700, 50)
            cover = st.slider("Overlap (characters)", 0, 300, 120, 10)
        if st.button("Index uploaded PDFs", type="primary", disabled=not files):
            emb = embedder_ready()
            with st.status("Indexing...", expanded=True) as status:
                bar = st.progress(0.0)
                for f in files:
                    st.write(f"**{f.name}**")
                    res = rag.ingest_pdf(storage, store, emb, f.getvalue(), f.name, force_ocr=force_ocr,
                                         chunk_size=csize, overlap=cover,
                                         progress=lambda m, fr=None: (st.write(m) if fr in (0.05, 0.25) else None,
                                                                      bar.progress(fr) if fr is not None else None))
                    if res["status"] == "indexed":
                        st.success(f"{res['pages']} pages → {res['chunks']} passages ({res.get('ocr_pages', 0)} via OCR)")
                    elif res["status"] == "already_indexed":
                        st.info(f"Already indexed ({res['chunks']} passages). Skipped.")
                    else:
                        st.warning("No readable text found.")
                    for w in res.get("warnings", []):
                        st.warning(w)
                status.update(label="Indexing finished", state="complete")
            st.session_state.uploader_key += 1
            st.rerun()
    with right:
        st.markdown("#### 🗂️ Choose from stored files")
        reg = storage.load_registry()
        if not reg:
            st.caption("No stored PDFs yet.")
        else:
            labels = {h: f"{e['name']}  ({e['pages']} pages)" for h, e in reg.items()}
            chosen = st.multiselect("Select files", options=list(labels), format_func=lambda h: labels[h])
            b1, b2 = st.columns(2)
            if b1.button("Re-index selected", disabled=not chosen):
                emb = embedder_ready()
                with st.status("Re-indexing...", expanded=True):
                    for h in chosen:
                        p = storage.pdf_path(h)
                        if p is None:
                            st.warning(f"{reg[h]['name']}: stored PDF missing, upload it again.")
                            continue
                        res = rag.ingest_pdf(storage, store, emb, p.read_bytes(), reg[h]["name"], force_ocr=force_ocr, reindex=True)
                        st.write(f"{reg[h]['name']}: {res['chunks']} passages")
                st.rerun()
            if b2.button("Delete selected", disabled=not chosen):
                for h in chosen:
                    rag.remove_document(storage, store, h)
                st.rerun()

    st.markdown("#### 📋 Indexed documents")
    if reg:
        rows = [{"Document": e["name"], "Pages": e["pages"], "OCR pages": e.get("ocr_pages", 0),
                 "Passages": e.get("indexed", {}).get(EMBED_NAME, {}).get("chunks", "not indexed"),
                 "Added": e.get("uploaded_at", "")} for e in reg.values()]
        st.dataframe(pd.DataFrame(rows), hide_index=True)
    st.markdown("#### 💾 Backup and restore")
    st.caption("Everything (PDFs, vector index, registry) lives in one folder. Download a backup so you can restore it after a Colab reset.")
    k1, k2 = st.columns(2)
    with k1:
        if st.button("Prepare backup ZIP"):
            z = storage.make_backup(Path(STORAGE_DIR).parent / "PalgharAI_backup.zip")
            st.session_state.backup_bytes = z.read_bytes()
        if st.session_state.get("backup_bytes"):
            st.download_button("⬇️ Download backup", st.session_state.backup_bytes, "PalgharAI_backup.zip", "application/zip")
    with k2:
        up = st.file_uploader("Restore from backup ZIP", type=["zip"], key="restore")
        if up and st.button("Restore now"):
            try:
                n = storage.restore_backup(up.getvalue())
                st.cache_resource.clear()
                st.success(f"Restored {n} files. Reloading...")
                st.rerun()
            except Exception as exc:
                st.error(f"Restore failed: {exc}")


def embedder_name_ok(entry):
    return EMBED_NAME in entry.get("indexed", {})


# ================================================================== PAGE 3: MODEL & SYSTEM
def kv(rows):
    return "".join(f'<div class="kv"><span>{html.escape(str(a))}</span><span>{html.escape(str(b))}</span></div>' for a, b in rows)


def page_system():
    hero("Live specification of the models and database that power this page.")
    try:
        emb = get_embedder(EMBED_NAME)
        espec = emb.spec()
    except Exception as exc:
        espec = {"name": EMBED_NAME, "dimension": "not loaded", "max_tokens": "-", "device": f"error: {exc}"[:60], "query_prefix": "-"}
    reg = storage.load_registry()
    c1, c2, c3 = st.columns(3)
    c1.markdown(f'<div class="card"><h4>🧠 Answer generator (LLM)</h4><div class="sub">Writes the answer from retrieved passages</div>'
                + kv([("Model", model_name), ("Provider", "Google Gemini API"), ("Temperature", temperature),
                      ("Status", "connected" if llm else "no API key"), ("Grounding", "context-only prompt"),
                      ("Citations", "[n] per fact")]) + "</div>", unsafe_allow_html=True)
    c2.markdown(f'<div class="card"><h4>🔎 Embedding model</h4><div class="sub">Turns Marathi / English text into vectors</div>'
                + kv([("Model", espec["name"]), ("Vector size", espec["dimension"]), ("Max tokens", espec["max_tokens"]),
                      ("Device", espec["device"]), ("Prefixes", espec["query_prefix"]), ("Languages", "100+ incl. Marathi, Hindi")])
                + "</div>", unsafe_allow_html=True)
    c3.markdown(f'<div class="card"><h4>🗄️ Vector database</h4><div class="sub">Stores passages and finds the closest ones</div>'
                + kv([("Engine", f"ChromaDB {store.chromadb_version}"), ("Index", "HNSW, cosine distance"),
                      ("Collection", store.name), ("Passages", store.count()), ("Documents", len(reg)),
                      ("Persistent", "yes (on disk)")]) + "</div>", unsafe_allow_html=True)

    st.markdown("#### How a question is answered")
    steps = [("1", "Question"), ("2", "Embed query"), ("3", "Search ChromaDB"), ("4", "Top passages"), ("5", "Gemini writes answer"), ("6", "Answer + sources")]
    cols = st.columns(6)
    for col, (n, t) in zip(cols, steps):
        col.markdown(f'<div class="step"><b>STEP {n}</b>{t}</div>', unsafe_allow_html=True)
    st.write("")

    last = next((m for m in reversed(st.session_state.messages) if m["role"] == "assistant" and m.get("meta")), None)
    a, b = st.columns(2)
    with a:
        st.markdown("#### Runtime")
        try:
            import torch
            gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
            tv = torch.__version__
        except Exception:
            gpu, tv = "unknown", "-"
        rows = [("Python", platform.python_version()), ("PyTorch", tv), ("Compute", gpu), ("Streamlit", st.__version__),
                ("Storage folder", STORAGE_DIR)]
        ok, langs = rag.ocr_available()
        rows.append(("OCR (Tesseract)", ("ready: " + ", ".join(l for l in langs if l in ("mar", "eng", "hin"))) if ok else "not installed"))
        st.markdown('<div class="card">' + kv(rows) + "</div>", unsafe_allow_html=True)
    with b:
        st.markdown("#### Last answer")
        if last:
            m = last["meta"]
            st.markdown('<div class="card">' + kv([("Search time", f"{m['retrieval_ms']:.0f} ms"),
                                                    ("Generation time", f"{m['generation_ms']:.0f} ms"),
                                                    ("Passages used", len(last["sources"])), ("Mode", m["mode"])]) + "</div>",
                        unsafe_allow_html=True)
        else:
            st.markdown('<div class="card"><div class="sub">Ask a question to see timings here.</div></div>', unsafe_allow_html=True)
    st.info("This system does not train a language model. Knowledge comes from your PDFs, which are indexed once and saved. "
            "Answers are written only from retrieved passages and always list their sources. Verify important figures in the original PDF.")


PAGES = {"💬 Ask Palghar AI": page_ask, "📚 Knowledge Base": page_kb, "⚙️ Model & System": page_system}
PAGES[st.session_state.page]()
st.divider()
st.caption("Palghar AI · RAG with ChromaDB · Academic project demonstration · AI can make mistakes, check the cited sources.")

Writing /content/app.py


In [6]:
os.makedirs("/content/.streamlit", exist_ok=True)
Path("/content/.streamlit/config.toml").write_text('''
[theme]
base = "dark"
primaryColor = "#17C3B2"
backgroundColor = "#0B1220"
secondaryBackgroundColor = "#111C2E"
textColor = "#E5E7EB"
font = "sans serif"

[server]
maxUploadSize = 200
''', encoding="utf-8")
print("App and theme written.")

App and theme written.


## Step 6 — Gemini API key (Colab Secrets)
The key is read from **Secrets → `GEMINI_API_KEY`** exactly like in the faculty's notebooks. It is never printed or saved in the notebook.

In [7]:
GEMINI_API_KEY = ""
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass

if not GEMINI_API_KEY and ASK_FOR_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Gemini API key (press Enter to skip and use retrieval-only mode): ").strip()

if GEMINI_API_KEY:
    try:
        from palghar_rag import GeminiLLM
        llm_test = GeminiLLM(GEMINI_API_KEY, model=GEMINI_MODEL)
        print("Gemini reply test:", llm_test.generate("Reply with the single word: OK")[:40])
    except Exception as exc:
        print(f"Gemini test failed: {type(exc).__name__}: {str(exc)[:200]}")
        try:
            print("Models available to your key:", ", ".join(GeminiLLM(GEMINI_API_KEY).list_models()[:15]))
            print("-> put one of these names in GEMINI_MODEL (Step 2) or in the web page sidebar.")
        except Exception:
            print("Could not list models. Check that the key is correct.")
else:
    print("No key: the app will run in retrieval-only mode (matching passages with sources, no written answer).")

Gemini API key (press Enter to skip and use retrieval-only mode): ··········
Gemini reply test: OK


## Step 7 — Download the embedding model once
This only downloads the model files (about 1 GB for `e5-base`) so the web page starts quickly. It is not training.

In [8]:

# STEP 7 — Repair and download embedding model

import os
import shutil
import gc
import torch

# Make sure Hugging Face uses the same cache as the Streamlit app
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Remove potentially corrupted Hugging Face cache
print("Cleaning old embedding model cache...")

if os.path.exists(HF_CACHE):
    shutil.rmtree(HF_CACHE)

os.makedirs(HF_CACHE, exist_ok=True)

# Download a fresh, complete model snapshot
from huggingface_hub import snapshot_download

print("Downloading fresh embedding model...")

model_path = snapshot_download(
    repo_id=EMBEDDING_MODEL,
    force_download=True
)

print("Model downloaded successfully:", model_path)

# Load model from the downloaded local directory
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading embedding model...")

_m = SentenceTransformer(
    model_path,
    device=device
)

print("Embedding model ready!")
print("Model:", EMBEDDING_MODEL)
print("Vector size:", _m.get_sentence_embedding_dimension())
print("Device:", device)

# Test embedding generation
test_embedding = _m.encode(
    ["Palghar district development and healthcare facilities"],
    normalize_embeddings=True
)

print("Embedding test successful:", test_embedding.shape)

del _m
del test_embedding

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Embedding model is ready for Streamlit!")


Cleaning old embedding model cache...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Model downloaded successfully: /content/drive/MyDrive/Palghar_AI/hf_cache/hub/models--intfloat--multilingual-e5-base/snapshots/d128750597153bb5987e10b1c3493a34e5a4502a
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model ready!
Model: intfloat/multilingual-e5-base
Vector size: 768
Device: cuda


/tmp/ipykernel_548/482737718.py:46: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Vector size:", _m.get_sentence_embedding_dimension())


Embedding test successful: (1, 768)
Embedding model is ready for Streamlit!


## Step 8 — Launch the web page 🚀
Starts Streamlit and a free Cloudflare tunnel. **Open the link printed below.** Keep this cell running while you present.

In [9]:
import subprocess, time, re, os, sys, urllib.request
from pathlib import Path
from IPython.display import display, HTML

APP_LOG, TUNNEL_LOG = Path("/content/streamlit_palghar.log"), Path("/content/cloudflare_palghar.log")

# stop anything from an earlier run
for var in ("streamlit_process", "tunnel_process"):
    proc = globals().get(var)
    if proc is not None:
        try:
            proc.terminate(); proc.wait(timeout=8)
        except Exception:
            pass

env = dict(os.environ,
           PALGHAR_STORAGE=STORAGE_DIR, PALGHAR_EMBED=EMBEDDING_MODEL,
           GEMINI_API_KEY=GEMINI_API_KEY, GEMINI_MODEL=GEMINI_MODEL, HF_HOME=HF_CACHE)

log = open(APP_LOG, "w", encoding="utf-8")
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "/content/app.py", f"--server.port={PORT}", "--server.address=0.0.0.0",
     "--server.headless=true", "--server.fileWatcherType=none", "--server.enableCORS=false",
     "--server.enableXsrfProtection=false", "--browser.gatherUsageStats=false"],
    stdout=log, stderr=subprocess.STDOUT, env=env, cwd="/content")

ready = False
for _ in range(60):
    if streamlit_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3) as r:
            ready = r.status == 200
            if ready:
                break
    except Exception:
        time.sleep(1)
if not ready:
    print(APP_LOG.read_text(errors="replace")[-6000:])
    raise RuntimeError("Streamlit failed to start. See the log above.")
print("Streamlit is running.")

BINARY = Path("/usr/local/bin/cloudflared")
if not BINARY.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", str(BINARY))
    os.chmod(BINARY, 0o755)

if TUNNEL_LOG.exists():
    TUNNEL_LOG.unlink()
tunnel_process = subprocess.Popen([str(BINARY), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate",
                                   "--loglevel", "info", "--logfile", str(TUNNEL_LOG)],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

public_url = None
for _ in range(100):
    if TUNNEL_LOG.exists():
        found = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text(errors="replace"))
        if found:
            public_url = found[0]
            break
    time.sleep(1)
if not public_url:
    print(TUNNEL_LOG.read_text(errors="replace")[-3000:] if TUNNEL_LOG.exists() else "no tunnel log")
    raise RuntimeError("Could not create the public link. See the log above.")

display(HTML(f'''
<div style="background:linear-gradient(120deg,#0F3A4A,#14284A);padding:26px;border-radius:16px;border:1px solid #25507A;margin-top:12px;font-family:sans-serif">
<h2 style="color:#F6F4EE;margin:0">🏛️ Palghar AI is live</h2>
<p style="color:#B9C9DC">Upload PDFs in <b>Knowledge Base</b>, then ask questions in <b>Ask Palghar AI</b>.</p>
<a href="{public_url}" target="_blank" style="display:inline-block;background:#0D9488;color:white;padding:13px 24px;border-radius:9px;font-weight:bold;text-decoration:none">OPEN PALGHAR AI →</a>
<p style="color:#8FA3BC;font-size:12px;margin:14px 0 0">Keep this Colab cell running. If the page says "bad gateway", wait 10 seconds and refresh.</p></div>'''))
print("Link:", public_url)

Streamlit is running.


Link: https://sponsors-subaru-foto-quilt.trycloudflare.com


## Step 9 — Save a backup (Option A, only if you are not using Google Drive)
Run this after indexing, then keep the ZIP. After a reconnect set `RESTORE_BACKUP = True` in Step 2 and upload it. The same ZIP can be created from the web page (*Knowledge Base → Backup and restore*).

In [10]:
'''zip_path = storage.make_backup("/content/PalgharAI_backup.zip")
print("Backup:", zip_path, f"({zip_path.stat().st_size / 1024**2:.1f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass'''

'zip_path = storage.make_backup("/content/PalgharAI_backup.zip")\nprint("Backup:", zip_path, f"({zip_path.stat().st_size / 1024**2:.1f} MB)")\ntry:\n    from google.colab import files\n    files.download(str(zip_path))\nexcept Exception:\n    pass'

---
## 🎓 What to tell the judges / faculty
* **Problem:** district documents are long and in Marathi; officials need answers with proof.
* **Solution:** Retrieval-Augmented Generation. PDFs → text (OCR for scans) → Marathi-aware chunks → multilingual embeddings → ChromaDB. A question is embedded, the closest passages are retrieved, and Gemini writes an answer **only from those passages**, citing `[1] [2]`.
* **Trust:** every answer lists *file · page · match % · exact passage*. If nothing relevant is found the app says so instead of guessing.
* **Persistence:** the index is saved on disk and can be backed up or kept on Drive, so nothing is re-processed after a reconnect.

## Troubleshooting
* **Marathi text looks broken in sources** → the PDF uses a legacy font. In *Knowledge Base* tick **Force OCR** and index again.
* **"Gemini could not answer"** → check the key, or put a model name from the list printed in Step 6 into the sidebar.
* **No answer found although the PDF has it** → lower *Minimum match*, raise *Passages to retrieve*, or rephrase using words from the document.
* **Out of GPU memory while indexing** → use `intfloat/multilingual-e5-small` in Step 2.
* **Limits to be honest about:** OCR on poor scans can misread digits; always verify important figures in the original page cited in the sources.